# 05 — Ipoh static susceptibility
This retrospective assessment uses a 500 m pilot grid and a 1 January 2023 cutoff. Notebook 04 remains the official comparison. This notebook verifies and reuses the deployment export. A relative SVM margin is not an occurrence probability.

## 1. Model integrity and ordered feature schema
Verification checks input hashes, estimator type, 19 ordered predictors, threshold, fitted preprocessing and development-score parity. It excludes frozen held-out records from tuning and fitting.

In [1]:
import json, os, sys
from datetime import date
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'app/backend/export_susceptibility.py').is_file())
sys.path.insert(0, str(root / 'app/backend'))
from export_susceptibility import FEATURES, recover_export
from sample_static_pilot import sample
from score_static_pilot import publish
model, metadata = recover_export('static')
assert list(model.feature_names_in_) == metadata['features'] == FEATURES['static']
print(metadata['algorithm'], 'C =', metadata['selected_c'], 'threshold =', metadata['threshold'])
print('Development / held-out excluded:', metadata['development_rows'], metadata['excluded_heldout_rows'])
print('Fitted preprocessing:', model.named_steps['prep'])

balanced linear-kernel SVC C = 0.1 threshold = 0.0
Development / held-out excluded: 1308 660
Fitted preprocessing: ColumnTransformer(sparse_threshold=0,
                  transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(keep_empty_features=True,
                                                                strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['elevation_m', 'slope_deg', 'aspect_sin',
                                  'aspect_cos', 'profile_curvature_m_inv',
                                  'plan_curvature_m_inv',
                                  'topographic_wetness_index',
                                  'soil_clay_surface', 'soil_sand_surface',
                                  's...ce', 'soil_bdod_surface',
                                  'road_distance_m', 'waterway_distance_m

## 2. Source, units and cutoff
SRTM elevations are metres and slopes degrees. Curvature uses 30 m windows in UTM zone 47N; MERIT TWI uses 90 m elevation and upstream area in km². SoilGrids 0–5 cm texture is in percent and density in g/cm³. GLC_FCS30D uses 2022, the last completed year. Notebook 03 defines road and waterway distances in EPSG:3857 metres. The bundled OSM vectors lack a reliable export date and are skipped unless supplied with documented pre-cutoff provenance. Earth Engine is optional and must already be authenticated. Notebook 02 uses NDVI for 2022 and 2020, Hansen forest loss during 2018–2022 relative to baseline tree cover within 250 m, and MERIT HAND in metres. These are retrospective product estimates.

In [2]:
reference = date(2023, 1, 1)
osm_date = os.environ.get('PILOT_OSM_SNAPSHOT_DATE')
saved = root / 'app/backend/pilot_data/static_feature_coverage.json'
coverage = json.loads(saved.read_text(encoding='utf-8')) if saved.exists() else {}
reuse = coverage.get('earth_engine_sampled', False) and coverage.get('assessment_date') == reference.isoformat()
pilot = sample(reference, date.fromisoformat(osm_date) if osm_date else None,
               None if reuse else os.environ.get('EE_PROJECT_ID'), reuse_ee=reuse)

{
  "cells": 450,
  "grid_sha256": "7a2814d9ede89abb65d08b16b62847c5e84cf6e08595cea03b5afb994c7a6ad6",
  "feature_table_sha256": "2230728f77c6f7533ae9caac76103d04c7978d428ec807b4af83d2a9678d0d5f",
  "assessment_date": "2023-01-01",
  "osm_snapshot_date": "2022-01-01",
  "osm_source": "https://download.geofabrik.de/asia/malaysia-singapore-brunei-220101-free.shp.zip",
  "osm_archive_sha256": "a17c9ace7bb2c0a5af50ec30ad7a8d4b5bcb9a3642164479c1684f19544f13a8",
  "earth_engine_sampled": true,
  "diagnostics": {
    "curvature_status": {
      "MEASURED": 426,
      "FLAT_BELOW_0.5_DEG": 24
    },
    "soilgrids_clay_status": {
      "MEASURED": 239,
      "INVALID_TEXTURE_SUM": 211
    },
    "soilgrids_sand_status": {
      "MEASURED": 239,
      "INVALID_TEXTURE_SUM": 211
    },
    "soilgrids_silt_status": {
      "MEASURED": 239,
      "INVALID_TEXTURE_SUM": 211
    },
    "soilgrids_bdod_status": {
      "MEASURED": 237,
      "OUT_OF_RANGE": 213
    }
  },
  "complete_cells": 362,
  "

## 3. Predictor coverage and unavailable cells
The feature CSV identifies each cell, coordinates, observed values and missing feature names. Missing or unsupported inputs exclude the cell from scoring.

In [3]:
for name in FEATURES['static']:
    count = int(pilot[name].notna().sum())
    print(f'{name}: {count}/{len(pilot)} measured')
print('Complete:', int(pilot[FEATURES['static']].notna().all(axis=1).sum()))
print(pilot[['cell_id', 'missing_features']].head().to_string(index=False))

elevation_m: 450/450 measured
slope_deg: 450/450 measured
aspect_sin: 450/450 measured
aspect_cos: 450/450 measured
profile_curvature_m_inv: 426/450 measured
plan_curvature_m_inv: 426/450 measured
topographic_wetness_index: 450/450 measured
soil_clay_surface: 449/450 measured
soil_sand_surface: 449/450 measured
soil_silt_surface: 449/450 measured
soil_bdod_surface: 449/450 measured
road_distance_m: 450/450 measured
waterway_distance_m: 450/450 measured
landcover_pre_event_class: 450/450 measured
ndvi_pre_event: 449/450 measured
ndvi_change_2y: 443/450 measured
forest_loss_5y_250m_pct: 378/450 measured
merit_upstream_area_km2: 450/450 measured
merit_hand_m: 450/450 measured
Complete: 362
              cell_id missing_features
static-500-19491-1160                 
static-500-19491-1161                 
static-500-19491-1162                 
static-500-19491-1163                 
static-500-19491-1164                 


## 4. Inference, categories and geographic publication
Apply the fitted SVC only to cells with all required measurements and supported land-cover categories. The official margin cutoff of 0 defines Lower (< 0) and Higher (≥ 0). The exporter validates grid identity, cell ID uniqueness, polygon validity, coordinate containment and feature provenance hashes.

In [4]:
assessment = publish()
print(json.dumps(assessment['assessment'], indent=2))
assert sum(feature['properties']['score'] is not None for feature in assessment['features']) == assessment['assessment']['scoredCells']

362 of 450 static pilot cells scored.
{
  "assessmentDate": "2023-01-01",
  "processedAt": "2026-11-25T19:16:21.881859+00:00",
  "model": "balanced linear-kernel SVC",
  "modelVersion": "d0c88d6dd05b",
  "modelSha256": "d0c88d6dd05b4e66c7e9e26dd2e59b36d64e59cac448a56a2b5589cc7446b358",
  "scoredCells": 362,
  "totalCells": 450,
  "threshold": 0.0,
  "rule": "Lower: score < 0; Higher: score \u2265 0.",
  "landcoverEncoding": "Unambiguous integer GLC codes matched to the fitted Notebook 04 labels.",
  "gridSha256": "7a2814d9ede89abb65d08b16b62847c5e84cf6e08595cea03b5afb994c7a6ad6",
  "featureTableSha256": "2230728f77c6f7533ae9caac76103d04c7978d428ec807b4af83d2a9678d0d5f",
  "osmSource": "https://download.geofabrik.de/asia/malaysia-singapore-brunei-220101-free.shp.zip",
  "osmSnapshotDate": "2022-01-01",
  "osmArchiveSha256": "a17c9ace7bb2c0a5af50ec30ad7a8d4b5bcb9a3642164479c1684f19544f13a8",
  "earthEngineSampled": true
}


## 5. Dashboard integration
FastAPI serves the validated GeoJSON at `/api/static-assessment`. In the Explorer, the Ipoh pilot view opens at local scale. Click an outlined cell for measured inputs and missingness. The Malaysia inventory button returns to the national view. Only cells with actual decision margins receive susceptibility colours.

In [5]:
from urllib.request import urlopen
from urllib.error import URLError
try:
    with urlopen('http://127.0.0.1:5173/api/static-assessment', timeout=3) as response:
        served = json.load(response)
    assert served['assessment']['scoredCells'] == assessment['assessment']['scoredCells']
    print('Vite/FastAPI integration:', len(served['features']), 'cells served')
except URLError as error:
    print('Live integration unavailable in this run:', error.reason)

Vite/FastAPI integration: 450 cells served


## 6. Malaysia coverage and resumable batches
The country grid uses the existing Malaysia boundary and 10 km equal-area cells. Separate tiles contain at most 100 cells. Coastal and island cells are clipped to the boundary and sampled at an interior point. Tile availability and missing predictors are reported independently of the score. A 50 km distance flag describes proximity to development locations and does not establish field validity. The finer 500 m grid is supported for bounded areas.
    
The verified January 2022 Geofabrik shapefiles supply road and waterway distances before the 1 January 2023 cutoff. The same model, feature schema, fixed threshold and Earth Engine source definitions are used throughout. Resume scripts and coverage provenance are documented in the application changelog.

In [6]:
from national_static import NATIONAL, summary
for resolution in (10000, 500):
    directory = NATIONAL / f'{reference.isoformat()}-{resolution}'
    manifest = directory / 'manifest.json'
    if manifest.exists():
        report = summary(directory, json.loads(manifest.read_text(encoding='utf-8')))
        keys = ['cellMetres', 'plannedCells', 'processedCells', 'unprocessedCells',
                'scoredCells', 'unscoredProcessedCells', 'cellsOver50KmFromDevelopment', 'modelVersion']
        print(json.dumps({key: report[key] for key in keys}, indent=2))

{
  "cellMetres": 10000,
  "plannedCells": 3827,
  "processedCells": 3827,
  "unprocessedCells": 0,
  "scoredCells": 2776,
  "unscoredProcessedCells": 1051,
  "cellsOver50KmFromDevelopment": 441,
  "modelVersion": "d0c88d6dd05b"
}
{
  "cellMetres": 500,
  "plannedCells": 285,
  "processedCells": 285,
  "unprocessedCells": 0,
  "scoredCells": 213,
  "unscoredProcessedCells": 72,
  "cellsOver50KmFromDevelopment": 0,
  "modelVersion": "d0c88d6dd05b"
}
